# 03 — Walk-Forward Backtest: Full Metric Suite

Evaluates all trained models across the four walk-forward periods using
**six metrics** reported for every algorithm and every period.

---

## Metrics reported

| Metric | What it measures | Why it matters for NSE |
|--------|-----------------|----------------------|
| **Sharpe ratio** | Excess return per unit of total risk | Primary proposal metric — will be negative for most strategies given 13% T-bill + 4.16% costs |
| **Sortino ratio** | Excess return per unit of *downside* risk only | More appropriate than Sharpe for fat-tailed NSE returns — upside swings don't penalise you |
| **Max drawdown** | Largest peak-to-trough loss | Most important for retail investors who cannot recover from large losses |
| **Calmar ratio** | Annualised return / max drawdown | Rewards agents that earn returns without catastrophic losses |
| **Total return %** | Raw portfolio return over the episode | Baseline performance before risk adjustment |
| **Win rate %** | % of days with positive return | Simple, intuitive metric for the retail investor audience |

---

## Correct Sharpe computation — per episode, not averaged series

**Wrong:**
```python
avg_vals = np.mean([ep1, ep2, ep3, ...], axis=0)   # averages out variance
sharpe   = compute_sharpe(avg_vals)                  # denominator too small -> inflated
```

**Correct (what this notebook does):**
```python
sharpes = [episode_sharpe(ep) for ep in episodes]   # per episode
mean_sharpe = np.mean(sharpes)                       # true expected Sharpe
```

---

## NSE context — why negative Sharpe is a finding, not a failure

| Item | Value |
|---|---|
| Kenya T-bill (risk-free rate) | ~13% / year = 0.000487 / day |
| NSE round-trip cost | 4.16% |
| Gross return needed for Sharpe > 0 | **> 17.2% / year** |
| NSE 20 Index historical average | 5–8% / year |

The meaningful comparisons are:
1. **Agent vs equal-weight baseline** — does the agent add value over a simple strategy?
2. **Sharpe generalisation gap < 0.10** — is the strategy robust to unseen data?


In [ ]:
import os, sys, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, "..")

from stable_baselines3 import PPO, DQN, A2C
from environment.nse_env  import NSEPortfolioEnv
from evaluation.metrics   import (
    episode_sharpe,
    mean_episode_sharpe,
    sharpe_generalisation_gap,
    sortino_ratio,
    max_drawdown,
    calmar_ratio,
    all_metrics,
    sharpe_context_nse,
    RF_DAILY,
)

os.makedirs("plots", exist_ok=True)

ctx = sharpe_context_nse()
print("NSE Sharpe Context")
print("=" * 55)
for k, v in ctx.items():
    print(f"  {k:<40}: {v}")

## Load best model for each algorithm

Loads `best_by_sharpe.zip` — the checkpoint saved when validation Sharpe
was highest during training. Falls back to the most recent final model
if the Sharpe-selected checkpoint does not exist yet.


In [ ]:
ALGOS   = ["ppo", "dqn", "a2c", "reinforce"]
PERIODS = ["train", "val", "test1", "test2"]
PERIOD_LABELS = {
    "train": "Train (07-17)",
    "val":   "Val (18-20)",
    "test1": "Test1 (21-22)",
    "test2": "Test2 (23-25)",
}


def load_best(algo):
    sharpe_path = f"models/{algo}/best_by_sharpe.zip"
    if os.path.exists(sharpe_path):
        print(f"  {algo:<12}: loaded best_by_sharpe.zip")
        p = sharpe_path.replace(".zip", "")
    else:
        finals = sorted(glob.glob(f"models/{algo}/*_final.zip"))
        if not finals:
            print(f"  {algo:<12}: NO MODEL FOUND — run 02_train.ipynb first")
            return None
        p = finals[-1].replace(".zip", "")
        print(f"  {algo:<12}: fallback -> {os.path.basename(p)}")
    if algo in ("ppo", "reinforce"):
        return PPO.load(p)
    elif algo == "dqn":
        return DQN.load(p)
    else:
        return A2C.load(p)


models = {a: load_best(a) for a in ALGOS}
models = {a: m for a, m in models.items() if m is not None}
print(f"\n{len(models)} models loaded.")

## Evaluation function — all six metrics per period

Runs `n_episodes` episodes. For each episode collects the full value
history and computes all metrics from it. Reports mean across episodes.

`domain_randomise=False` keeps costs fixed at calibrated values
so results are reproducible.


In [ ]:
def evaluate_period(model, period, n_episodes=20):
    """
    Run n_episodes on one walk-forward period.
    Returns mean of each metric across all episodes.
    """
    env = NSEPortfolioEnv(period=period, domain_randomise=False)

    ep_sharpe   = []
    ep_sortino  = []
    ep_mdd      = []
    ep_calmar   = []
    ep_return   = []
    ep_winrate  = []
    ep_values   = []   # store all value histories for mean_episode_sharpe()

    for _ in range(n_episodes):
        obs, _ = env.reset()
        done   = False
        values = [env.INITIAL_CAPITAL]

        while not done:
            act, _ = model.predict(obs, deterministic=True)
            obs, _, term, trunc, info = env.step(int(act))
            done = term or trunc
            values.append(info["portfolio_value"])

        ep_values.append(values)

        # Compute all metrics for this individual episode
        v   = np.array(values)
        r   = np.diff(v) / (v[:-1] + 1e-8)

        ep_sharpe.append(episode_sharpe(values, RF_DAILY))
        ep_sortino.append(sortino_ratio(values, RF_DAILY))
        ep_mdd.append(max_drawdown(values) * 100)
        ep_calmar.append(calmar_ratio(values))
        ep_return.append(info["total_return_pct"])
        ep_winrate.append(float(np.mean(r > 0) * 100))

    env.close()

    # Sharpe std for uncertainty reporting
    sh_stats = mean_episode_sharpe(ep_values, RF_DAILY)

    return {
        "sharpe_mean":   round(float(np.mean(ep_sharpe)),  4),
        "sharpe_std":    round(float(np.std(ep_sharpe)),   4),
        "sharpe_pct_positive": round(sh_stats["pct_positive"], 1),
        "sortino_mean":  round(float(np.mean(ep_sortino)), 4),
        "sortino_std":   round(float(np.std(ep_sortino)),  4),
        "mdd_mean":      round(float(np.mean(ep_mdd)),     2),
        "mdd_std":       round(float(np.std(ep_mdd)),      2),
        "calmar_mean":   round(float(np.mean(ep_calmar)),  4),
        "return_mean":   round(float(np.mean(ep_return)),  4),
        "return_std":    round(float(np.std(ep_return)),   4),
        "winrate_mean":  round(float(np.mean(ep_winrate)), 2),
        "n_episodes":    n_episodes,
    }


print("Evaluation function ready.")
print(f"Will run 20 episodes x {len(PERIODS)} periods x {len(models)} algorithms.")

## Run walk-forward evaluation across all periods

In [ ]:
all_results = {}

for algo, model in models.items():
    print(f"\n{'='*55}")
    print(f"  {algo.upper()}")
    print(f"{'='*55}")
    all_results[algo] = {}

    for period in PERIODS:
        r = evaluate_period(model, period, n_episodes=20)
        all_results[algo][period] = r
        print(
            f"  {period:<7}  "
            f"Sharpe {r['sharpe_mean']:>+7.4f} \u00b1{r['sharpe_std']:.3f}  "
            f"Sortino {r['sortino_mean']:>+7.4f}  "
            f"MDD {r['mdd_mean']:>5.1f}%  "
            f"Return {r['return_mean']:>+7.2f}%  "
            f"WinRate {r['winrate_mean']:>5.1f}%"
        )

## Equal-weight baseline

Buys an equal share of all available stocks, holds for 252 days.
NSE entry cost (2.08% one-way) is deducted at purchase.
All six metrics computed for the baseline too.


In [ ]:
from data.database         import get_db, N_STOCKS
from environment.nse_costs import NSECostModel


def equal_weight_baseline(period, n_episodes=20):
    db     = get_db()
    costs  = NSECostModel()
    data   = db.get_period(period)
    prices = data["prices"]
    avail  = data["availability"]
    T      = data["T"]

    ep_sharpe  = []; ep_sortino = []; ep_mdd    = []
    ep_calmar  = []; ep_return  = []; ep_winrate = []

    for _ in range(n_episodes):
        start = np.random.randint(65, max(66, T - 252))
        end   = min(start + 252, T)
        av    = avail[start]
        na    = int(av.sum())
        if na == 0:
            continue
        w = av / na

        capital = 100_000.0
        buy_cost = sum(
            costs.compute_cost(capital * w[i])
            for i in range(N_STOCKS) if w[i] > 0
        )
        capital -= buy_cost
        values = [capital]

        for t in range(start + 1, end):
            ret     = ((prices[t] - prices[t-1]) / (prices[t-1] + 1e-8)) * avail[t]
            capital *= (1 + float(np.dot(w, ret)))
            values.append(capital)

        v = np.array(values)
        r = np.diff(v) / (v[:-1] + 1e-8)

        ep_sharpe.append(episode_sharpe(values, RF_DAILY))
        ep_sortino.append(sortino_ratio(values, RF_DAILY))
        ep_mdd.append(max_drawdown(values) * 100)
        ep_calmar.append(calmar_ratio(values))
        ep_return.append((capital / 100_000.0 - 1) * 100)
        ep_winrate.append(float(np.mean(r > 0) * 100))

    return {
        "sharpe_mean":  round(float(np.mean(ep_sharpe)),  4),
        "sortino_mean": round(float(np.mean(ep_sortino)), 4),
        "mdd_mean":     round(float(np.mean(ep_mdd)),     2),
        "calmar_mean":  round(float(np.mean(ep_calmar)),  4),
        "return_mean":  round(float(np.mean(ep_return)),  4),
        "winrate_mean": round(float(np.mean(ep_winrate)), 2),
    }


print("Equal-weight baseline (buy-and-hold with NSE entry costs):")
baselines = {}
for p in PERIODS:
    b = equal_weight_baseline(p)
    baselines[p] = b
    print(
        f"  {p:<7}  "
        f"Sharpe {b['sharpe_mean']:>+7.4f}  "
        f"Sortino {b['sortino_mean']:>+7.4f}  "
        f"MDD {b['mdd_mean']:>5.1f}%  "
        f"Return {b['return_mean']:>+7.2f}%  "
        f"WinRate {b['winrate_mean']:>5.1f}%"
    )

## Full summary tables

One table per metric, all algorithms and all periods.
Baseline shown at the bottom of each table for comparison.


In [ ]:
def print_metric_table(metric_key, metric_name, fmt="{:>+9.4f}"):
    print(f"\n{'─'*75}")
    print(f"  {metric_name}")
    print(f"{'─'*75}")
    header = f"  {'Algorithm':<12}"
    for p in PERIODS:
        header += f"  {PERIOD_LABELS[p]:>16}"
    print(header)
    print(f"  {'─'*72}")

    for algo, res in all_results.items():
        row = f"  {algo.upper():<12}"
        for p in PERIODS:
            val = res[p][metric_key]
            row += f"  {fmt.format(val):>16}"
        print(row)

    # Baseline
    row = f"  {'EqWeight':<12}"
    for p in PERIODS:
        val = baselines[p][metric_key]
        row += f"  {fmt.format(val):>16}"
    print(f"  {'─'*72}")
    print(row)


print_metric_table("sharpe_mean",  "SHARPE RATIO  (higher is better, > 0 beats T-bill)")
print_metric_table("sortino_mean", "SORTINO RATIO  (higher is better, penalises downside only)")
print_metric_table("mdd_mean",     "MAX DRAWDOWN %  (lower is better)", fmt="{:>9.2f}")
print_metric_table("calmar_mean",  "CALMAR RATIO  (higher is better)")
print_metric_table("return_mean",  "TOTAL RETURN %  (higher is better)", fmt="{:>+9.2f}")
print_metric_table("winrate_mean", "WIN RATE %  (% of days with positive return)", fmt="{:>9.2f}")

## Sharpe generalisation gap

**Proposal target: gap < 0.10**

```
gap = |train_sharpe - test1_sharpe| / |train_sharpe|
```

A gap below 0.10 means test Sharpe is within 10% of training Sharpe —
the agent learned a robust strategy rather than memorising training data.


In [ ]:
print("\nSharpe Generalisation Gap  (train -> test1)")
print("=" * 65)
gap_results = {}
for algo in all_results:
    tr = all_results[algo]["train"]["sharpe_mean"]
    t1 = all_results[algo]["test1"]["sharpe_mean"]
    g  = sharpe_generalisation_gap(tr, t1)
    gap_results[algo] = g
    status = "\u2713 PASS" if g["target_met"] else "\u2717 FAIL"
    print(
        f"  {algo.upper():<12}  "
        f"train = {tr:>+.4f}   "
        f"test1 = {t1:>+.4f}   "
        f"gap = {g['normalised_gap']:.4f}   [{status}]"
    )
    print(f"             {g['interpretation']}")

## Beat-the-baseline summary

For each algorithm and each period, shows whether it beat the equal-weight
baseline on each metric. A + means it outperformed the baseline.


In [ ]:
print("\nBeat equal-weight baseline?  (+ = yes, - = no)")
print("=" * 75)

metrics_to_check = [
    ("sharpe_mean",  "Sharpe",  True),   # True = higher is better
    ("sortino_mean", "Sortino", True),
    ("mdd_mean",     "MDD",     False),  # False = lower is better
    ("return_mean",  "Return",  True),
]

header = f"  {'Algorithm':<12}  {'Period':<8}"
for _, mname, _ in metrics_to_check:
    header += f"  {mname:>8}"
print(header)
print("  " + "─" * 60)

for algo, res in all_results.items():
    for period in ["test1", "test2"]:
        row = f"  {algo.upper():<12}  {period:<8}"
        for mkey, mname, higher_better in metrics_to_check:
            agent_val    = res[period][mkey]
            baseline_val = baselines[period][mkey]
            if higher_better:
                beat = agent_val > baseline_val
            else:
                beat = agent_val < baseline_val
            row += f"  {'  +   ' if beat else '  -   ':>8}"
        print(row)
    print("  " + "─" * 60)

## Plots

### Plot 1 — Sharpe ratio across periods with std error bars
### Plot 2 — Sortino ratio across periods
### Plot 3 — Maximum drawdown across periods
### Plot 4 — Total return across periods
### Plot 5 — Sharpe generalisation gap (pass/fail)
### Plot 6 — Win rate across periods


In [ ]:
colors  = ["#1f77b4","#ff7f0e","#2ca02c","#d62728"]
algos   = list(all_results.keys())

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

x = np.arange(4)
w = 0.18

# ── Plot 1: Sharpe ─────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for i, algo in enumerate(algos):
    vals = [all_results[algo][p]["sharpe_mean"] for p in PERIODS]
    errs = [all_results[algo][p]["sharpe_std"]  for p in PERIODS]
    ax1.bar(x + i*w, vals, w, yerr=errs, capsize=3,
            label=algo.upper(), color=colors[i], alpha=0.85)
base = [baselines[p]["sharpe_mean"] for p in PERIODS]
ax1.step(np.append(x - 0.1, x[-1] + len(algos)*w + 0.1),
         np.append(base, base[-1]),
         where="post", color="black", lw=1.5, ls="--", label="Equal-weight")
ax1.axhline(0, color="red", lw=0.8, ls=":", label="Sharpe=0 (T-bill)")
ax1.set_xticks(x + w*1.5)
ax1.set_xticklabels(["Train\n07-17","Val\n18-20","Test1\n21-22","Test2\n23-25"])
ax1.set_title("Sharpe ratio (\u00b1 std)")
ax1.set_ylabel("Sharpe ratio")
ax1.legend(fontsize=7)

# ── Plot 2: Sortino ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
for i, algo in enumerate(algos):
    vals = [all_results[algo][p]["sortino_mean"] for p in PERIODS]
    ax2.bar(x + i*w, vals, w, label=algo.upper(), color=colors[i], alpha=0.85)
base = [baselines[p]["sortino_mean"] for p in PERIODS]
ax2.step(np.append(x - 0.1, x[-1] + len(algos)*w + 0.1),
         np.append(base, base[-1]),
         where="post", color="black", lw=1.5, ls="--", label="Equal-weight")
ax2.axhline(0, color="red", lw=0.8, ls=":")
ax2.set_xticks(x + w*1.5)
ax2.set_xticklabels(["Train\n07-17","Val\n18-20","Test1\n21-22","Test2\n23-25"])
ax2.set_title("Sortino ratio (downside risk only)")
ax2.set_ylabel("Sortino ratio")
ax2.legend(fontsize=7)

# ── Plot 3: Max Drawdown ────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
for i, algo in enumerate(algos):
    vals = [all_results[algo][p]["mdd_mean"] for p in PERIODS]
    ax3.bar(x + i*w, vals, w, label=algo.upper(), color=colors[i], alpha=0.85)
base = [baselines[p]["mdd_mean"] for p in PERIODS]
ax3.step(np.append(x - 0.1, x[-1] + len(algos)*w + 0.1),
         np.append(base, base[-1]),
         where="post", color="black", lw=1.5, ls="--", label="Equal-weight")
ax3.set_xticks(x + w*1.5)
ax3.set_xticklabels(["Train\n07-17","Val\n18-20","Test1\n21-22","Test2\n23-25"])
ax3.set_title("Max drawdown % (lower is better)")
ax3.set_ylabel("Max drawdown (%)")
ax3.legend(fontsize=7)

# ── Plot 4: Return ─────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
for i, algo in enumerate(algos):
    vals = [all_results[algo][p]["return_mean"] for p in PERIODS]
    errs = [all_results[algo][p]["return_std"]  for p in PERIODS]
    ax4.bar(x + i*w, vals, w, yerr=errs, capsize=3,
            label=algo.upper(), color=colors[i], alpha=0.85)
base = [baselines[p]["return_mean"] for p in PERIODS]
ax4.step(np.append(x - 0.1, x[-1] + len(algos)*w + 0.1),
         np.append(base, base[-1]),
         where="post", color="black", lw=1.5, ls="--", label="Equal-weight")
ax4.axhline(0, color="red", lw=0.8, ls=":")
ax4.set_xticks(x + w*1.5)
ax4.set_xticklabels(["Train\n07-17","Val\n18-20","Test1\n21-22","Test2\n23-25"])
ax4.set_title("Total return % (\u00b1 std)")
ax4.set_ylabel("Return (%)")
ax4.legend(fontsize=7)

# ── Plot 5: Generalisation Gap ──────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
gaps       = [gap_results[a]["normalised_gap"] for a in algos]
bar_colors = ["#2ca02c" if g < 0.10 else "#d62728" for g in gaps]
bars = ax5.bar(algos, [g * 100 for g in gaps], color=bar_colors)
ax5.axhline(10, color="orange", lw=1.5, ls="--", label="Target: 10%")
ax5.set_ylabel("Normalised Sharpe gap (%)")
ax5.set_title("Sharpe generalisation gap: train \u2192 test1\n(green = PASS, red = FAIL)")
ax5.legend()
for bar, g in zip(bars, gaps):
    ax5.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f"{g:.3f}", ha="center", va="bottom", fontsize=9)

# ── Plot 6: Win Rate ────────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 1])
for i, algo in enumerate(algos):
    vals = [all_results[algo][p]["winrate_mean"] for p in PERIODS]
    ax6.bar(x + i*w, vals, w, label=algo.upper(), color=colors[i], alpha=0.85)
base = [baselines[p]["winrate_mean"] for p in PERIODS]
ax6.step(np.append(x - 0.1, x[-1] + len(algos)*w + 0.1),
         np.append(base, base[-1]),
         where="post", color="black", lw=1.5, ls="--", label="Equal-weight")
ax6.axhline(50, color="red", lw=0.8, ls=":", label="50% (random)")
ax6.set_xticks(x + w*1.5)
ax6.set_xticklabels(["Train\n07-17","Val\n18-20","Test1\n21-22","Test2\n23-25"])
ax6.set_title("Win rate % (days with positive return)")
ax6.set_ylabel("Win rate (%)")
ax6.legend(fontsize=7)

plt.suptitle(
    "NSE DRL Portfolio Management — Full Metric Suite\n"
    "Walk-forward evaluation across all algorithms and periods",
    fontsize=12, y=1.01
)
plt.savefig("plots/full_metric_suite.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: plots/full_metric_suite.png")

## Per-algorithm radar / spider chart

Shows all six metrics for each algorithm on test1 relative to the baseline.
Values above 1.0 on each axis mean the algorithm beat the baseline on that metric.


In [ ]:
from matplotlib.patches import FancyArrowPatch

metric_keys    = ["sharpe_mean","sortino_mean","mdd_mean","calmar_mean","return_mean","winrate_mean"]
metric_labels  = ["Sharpe","Sortino","MDD\n(lower=better)","Calmar","Return","Win Rate"]
higher_better  = [True, True, False, True, True, True]

fig, axes = plt.subplots(1, len(algos), figsize=(16, 4),
                          subplot_kw={"projection": "polar"})

angles = np.linspace(0, 2*np.pi, len(metric_keys), endpoint=False).tolist()
angles += angles[:1]

for ax, algo in zip(axes, algos):
    res_t1   = all_results[algo]["test1"]
    base_t1  = baselines["test1"]

    # Normalise each metric relative to baseline (baseline = 1.0)
    vals = []
    for mk, hb in zip(metric_keys, higher_better):
        agent_v = res_t1[mk]
        base_v  = base_t1[mk]
        if base_v == 0:
            norm = 1.0
        elif hb:
            norm = agent_v / base_v if base_v != 0 else 1.0
        else:
            # lower is better: flip so > 1 = better than baseline
            norm = base_v / agent_v if agent_v != 0 else 1.0
        vals.append(norm)

    vals += vals[:1]

    ax.plot(angles, vals, "o-", lw=1.5, color="#1f77b4")
    ax.fill(angles, vals, alpha=0.2, color="#1f77b4")
    ax.axhline(y=1.0, color="gray", lw=0.8, ls="--", alpha=0.5)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_labels, fontsize=7)
    ax.set_title(algo.upper(), pad=12)
    ax.set_ylim(0, 2)
    ax.axhline(y=1.0, color="gray", lw=0.8, ls="--")

plt.suptitle(
    "Test1 (2021-2022): each metric relative to equal-weight baseline\n"
    "Values > 1.0 = beat the baseline on that metric",
    fontsize=10
)
plt.tight_layout()
plt.savefig("plots/radar_vs_baseline.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: plots/radar_vs_baseline.png")

## Save all results

In [ ]:
output = {
    "algorithm_results":      all_results,
    "generalisation_gaps":    gap_results,
    "equal_weight_baselines": baselines,
    "nse_context":            sharpe_context_nse(),
}
with open("models/backtest_results.json", "w") as f:
    json.dump(output, f, indent=2, default=str)

print("Saved: models/backtest_results.json")
print()
print("=" * 55)
print("BACKTEST COMPLETE")
print("=" * 55)
print()
print("Metrics reported:")
print("  Sharpe ratio        — primary risk-adjusted metric")
print("  Sortino ratio       — downside risk only (more appropriate for NSE)")
print("  Max drawdown        — most important for retail investors")
print("  Calmar ratio        — return per unit of drawdown risk")
print("  Total return %      — raw performance")
print("  Win rate %          — % of profitable days")
print()
print("Charts saved:")
print("  plots/full_metric_suite.png   — 6-panel comparison")
print("  plots/radar_vs_baseline.png   — per-algorithm radar chart")
print()
print("Note: Negative Sharpe values are expected given NSE context.")
print("The meaningful comparisons are agent vs equal-weight baseline")
print("and the Sharpe generalisation gap vs the 0.10 target.")